# Cross-Validation — Variance Estimates for All Four Models
Adds 5-fold stratified cross-validation on top of the existing single-split results (BLS_baseline, RF_main, RF_with_robovetter, CNN_1D). Same features, same architecture, same hyperparameters as Weeks 3-4 — this only adds a variance estimate, not new tuning.

**Why this matters for the paper:** every number reported so far comes from one fixed 70/15/15 split. This notebook answers: is the RF_main vs CNN_1D gap real and stable, or partly an artifact of that one particular split?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DRIVE_ROOT = '/content/drive/MyDrive/SADA'
CSV_PATH   = os.path.join(DRIVE_ROOT, 'balanced_tce.csv')
NPY_DIR    = os.path.join(DRIVE_ROOT, 'processed')
CV_RESULTS_PATH = os.path.join(DRIVE_ROOT, 'cv_results.csv')

## 1. Rebuild the merged dataset + engineer flux features (same as Week 3)
One-time cost — everything below reuses this, no re-computation per fold.

In [ ]:
balanced = pd.read_csv(CSV_PATH)

npy_files = [f for f in os.listdir(NPY_DIR) if f.endswith('.npy') and f != 'failed_kepids.npy']
pattern = re.compile(r'^(\d+)_(\d+)_(PC|FP)\.npy$')
records = []
for fname in npy_files:
    m = pattern.match(fname)
    if not m:
        continue
    kepid, plnt_num, label_str = m.groups()
    records.append({
        'kepid': int(kepid), 'tce_plnt_num': int(plnt_num),
        'label': 1 if label_str == 'PC' else 0,
        'npy_path': os.path.join(NPY_DIR, fname)
    })
npy_index = pd.DataFrame(records)

data = pd.merge(npy_index, balanced, on=['kepid', 'tce_plnt_num'], how='inner')
data = data.dropna(subset=['tce_period','tce_duration','tce_depth','tce_model_snr']).reset_index(drop=True)
print(f"Row set: {len(data)}")

In [ ]:
def flux_shape_features(flux):
    flux = np.asarray(flux, dtype=float)
    n = len(flux)
    mean, std = flux.mean(), flux.std()
    minimum, maximum = flux.min(), flux.max()
    lo, hi = n // 3, 2 * n // 3
    in_transit = flux[lo:hi]
    out_transit = np.concatenate([flux[:lo], flux[hi:]])
    depth_est = out_transit.mean() - in_transit.min()
    in_std = in_transit.std()
    out_std = out_transit.std() if len(out_transit) else 0.0
    if std > 0:
        skew = ((flux - mean) ** 3).mean() / (std ** 3)
        kurt = ((flux - mean) ** 4).mean() / (std ** 4) - 3
    else:
        skew, kurt = 0.0, 0.0
    return {'flux_mean': mean, 'flux_std': std, 'flux_min': minimum, 'flux_max': maximum,
            'flux_depth_est': depth_est, 'flux_in_transit_std': in_std,
            'flux_out_transit_std': out_std, 'flux_skew': skew, 'flux_kurtosis': kurt}

# preload every flux array into memory ONCE (avoids repeated Drive I/O across 5 folds)
flux_matrix = []
feature_rows = []
for path in tqdm(data['npy_path'], desc='Preloading flux arrays'):
    flux = np.load(path).astype(np.float32)
    flux_matrix.append(flux)
    feature_rows.append(flux_shape_features(flux))

flux_matrix = np.stack(flux_matrix)     # shape (N, 200), in memory for the whole notebook
flux_feats = pd.DataFrame(feature_rows)
data = pd.concat([data.reset_index(drop=True), flux_feats], axis=1)
labels_all = data['label'].values.astype(np.float32)

BLS_FEATURES = ['tce_period', 'tce_duration', 'tce_depth', 'tce_model_snr']
FLUX_FEATURES = ['flux_mean', 'flux_std', 'flux_min', 'flux_max', 'flux_depth_est',
                 'flux_in_transit_std', 'flux_out_transit_std', 'flux_skew', 'flux_kurtosis']
RF_FEATURES = BLS_FEATURES + FLUX_FEATURES
RF_FEATURES_WITH_ROBOVETTER = RF_FEATURES + ['robovetter_score']

print(f"Preloaded {flux_matrix.shape[0]} flux arrays, engineered {flux_feats.shape[1]} features")

## 2. CNN architecture (identical to Week 4 — 8/16/32 channels, GAP head)

In [ ]:
class Conv1DClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv1d(1, 8, 5, padding=2), nn.BatchNorm1d(8),
                                     nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.block2 = nn.Sequential(nn.Conv1d(8, 16, 5, padding=2), nn.BatchNorm1d(16),
                                     nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.block3 = nn.Sequential(nn.Conv1d(16, 32, 5, padding=2), nn.BatchNorm1d(32),
                                     nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout))
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(32, 16), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(16, 1))

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.gap(x).squeeze(-1)
        return self.head(x).squeeze(-1)

class InMemoryFluxDataset(Dataset):
    """Reads from the preloaded flux_matrix by index — zero disk I/O."""
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        flux = torch.tensor(flux_matrix[idx]).unsqueeze(0)
        label = torch.tensor(labels_all[idx])
        return flux, label

def train_cnn(train_idx, val_idx, max_epochs=40, patience=6):
    model = Conv1DClassifier().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    train_loader = DataLoader(InMemoryFluxDataset(train_idx), batch_size=64, shuffle=True)
    val_loader = DataLoader(InMemoryFluxDataset(val_idx), batch_size=64, shuffle=False)

    best_val_loss, best_state, no_improve = float('inf'), None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(criterion(model(xb), yb).item())
        val_loss = np.mean(val_losses)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model

def predict_cnn(model, test_idx):
    model.eval()
    loader = DataLoader(InMemoryFluxDataset(test_idx), batch_size=64, shuffle=False)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            probs.extend(torch.sigmoid(model(xb)).cpu().numpy())
    return np.array(probs)

## 3. 5-fold stratified CV loop — all four models per fold

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
    }

fold_results = []

for fold_i, (train_idx, test_idx) in enumerate(skf.split(data, labels_all), start=1):
    print(f"\n========== Fold {fold_i}/{N_FOLDS} ==========")
    y_train_full, y_test = labels_all[train_idx], labels_all[test_idx]

    # ---- BLS_baseline (Logistic Regression) ----
    X_train = data.loc[train_idx, BLS_FEATURES].values
    X_test  = data.loc[test_idx,  BLS_FEATURES].values
    scaler = StandardScaler().fit(X_train)
    lr = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train_full)
    preds = lr.predict(scaler.transform(X_test))
    probs = lr.predict_proba(scaler.transform(X_test))[:, 1]
    m = compute_metrics(y_test, preds, probs); m.update(model='BLS_baseline', fold=fold_i)
    fold_results.append(m)
    print(f"  BLS_baseline    f1={m['f1']:.4f} auc={m['roc_auc']:.4f}")

    # ---- RF_main ----
    X_train = data.loc[train_idx, RF_FEATURES].values
    X_test  = data.loc[test_idx,  RF_FEATURES].values
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_train, y_train_full)
    preds = rf.predict(X_test)
    probs = rf.predict_proba(X_test)[:, 1]
    m = compute_metrics(y_test, preds, probs); m.update(model='RF_main', fold=fold_i)
    fold_results.append(m)
    print(f"  RF_main         f1={m['f1']:.4f} auc={m['roc_auc']:.4f}")

    # ---- RF_with_robovetter ----
    X_train = data.loc[train_idx, RF_FEATURES_WITH_ROBOVETTER].values
    X_test  = data.loc[test_idx,  RF_FEATURES_WITH_ROBOVETTER].values
    rf2 = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_train, y_train_full)
    preds = rf2.predict(X_test)
    probs = rf2.predict_proba(X_test)[:, 1]
    m = compute_metrics(y_test, preds, probs); m.update(model='RF_with_robovetter', fold=fold_i)
    fold_results.append(m)
    print(f"  RF_with_robovetter f1={m['f1']:.4f} auc={m['roc_auc']:.4f}")

    # ---- CNN_1D (inner train/val split within this fold's training data, for early stopping) ----
    inner_train_idx, inner_val_idx = train_test_split(
        train_idx, test_size=0.15, random_state=SEED, stratify=y_train_full)
    cnn_model = train_cnn(inner_train_idx, inner_val_idx)
    probs = predict_cnn(cnn_model, test_idx)
    preds = (probs > 0.5).astype(int)
    m = compute_metrics(y_test, preds, probs); m.update(model='CNN_1D', fold=fold_i)
    fold_results.append(m)
    print(f"  CNN_1D          f1={m['f1']:.4f} auc={m['roc_auc']:.4f}")

fold_results_df = pd.DataFrame(fold_results)
print("\nAll folds complete.")

## 4. Aggregate — mean ± std per model across folds
This is the table that goes in the paper: not just a single number, but a mean and spread that shows whether the RF vs CNN gap holds up across different data splits.

In [ ]:
summary = fold_results_df.groupby('model').agg(
    accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', 'std'),
    precision_mean=('precision', 'mean'), precision_std=('precision', 'std'),
    recall_mean=('recall', 'mean'), recall_std=('recall', 'std'),
    f1_mean=('f1', 'mean'), f1_std=('f1', 'std'),
    roc_auc_mean=('roc_auc', 'mean'), roc_auc_std=('roc_auc', 'std'),
).reset_index()

model_order = ['BLS_baseline', 'CNN_1D', 'RF_main', 'RF_with_robovetter']
summary['model'] = pd.Categorical(summary['model'], categories=model_order, ordered=True)
summary = summary.sort_values('model').reset_index(drop=True)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(summary[['model', 'f1_mean', 'f1_std', 'roc_auc_mean', 'roc_auc_std']])
summary

## 5. Save both per-fold and summary results to Drive

In [ ]:
fold_results_df.to_csv(os.path.join(DRIVE_ROOT, 'cv_fold_results.csv'), index=False)
summary.to_csv(CV_RESULTS_PATH, index=False)
print(f"Saved per-fold results and summary to {DRIVE_ROOT}")